In [43]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [5]:
df = pd.read_csv("../data/creditcard_processed.csv")

## 1. Train/test split (stratified)

In [29]:
X = df.drop(columns=['Amount', 'Class'])
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=321, stratify=y)

In [30]:
# Quick check that train & test sets have the same no. of rows
print(f"X_train: {len(X_train)}, y_train: {len(y_train)}")
print(f"X_test: {len(X_test)}, y_test: {len(y_test)}")

X_train: 227845, y_train: 227845
X_test: 56962, y_test: 56962


In [33]:
# Quick check that partitions are stratified as expected
print(y_test.value_counts() / (y_test.value_counts() + y_train.value_counts()))

Class
0    0.200004
1    0.199187
Name: count, dtype: float64


## 2. Train simple model with class weights

In [37]:
model = LogisticRegression(class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

## 3. Evaluate on test set

#### Target Metrics:

__Precision:__ >= 80%

__Recall:__ >= 5%

In [39]:
y_pred = model.predict(X_test)

In [40]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.95      0.97     56864
           1       0.03      0.84      0.06        98

    accuracy                           0.95     56962
   macro avg       0.51      0.89      0.52     56962
weighted avg       1.00      0.95      0.97     56962



In [41]:
print(confusion_matrix(y_test, y_pred))

[[54102  2762]
 [   16    82]]


## 4. Tune threshold

In [53]:
# Threshold tuning
from sklearn.metrics import precision_recall_curve

y_proba = model.predict_proba(X_test)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# Find threshold for 5% precision
idx = np.argmax(precisions >= 0.05)
print(f"At 5% precision: recall = {recalls[idx]:.2f}, threshold = {thresholds[idx]:.3f}")

At 5% precision: recall = 0.82, threshold = 0.632


Adjusting the classification threshold to 0.632 increases Precision to meet our target of 5%, while still meeting the target Recall of >=80%.

## 5. Re-evaluate

In [54]:
# Reclassify predictions based on new threshold
y_pred = (y_proba >= 0.632).astype(int)

In [55]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56864
           1       0.05      0.82      0.09        98

    accuracy                           0.97     56962
   macro avg       0.52      0.89      0.54     56962
weighted avg       1.00      0.97      0.98     56962



In [56]:
print(confusion_matrix(y_test, y_pred))

[[55346  1518]
 [   18    80]]


Adjusting the threshold to increase precision has meant 2 additional cases of fraud are "missed" (16 false negatives based on 0.5 threshold, 18 with new threshold). The number of false positives has reduced from 2672 to 1518. 

Depending on business priorities, this may or may not be an acceptable trade-off.